## Import Library

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os, json, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout,
    Bidirectional, GlobalMaxPool1D, Conv1D,
    MaxPooling1D, BatchNormalization, Flatten, GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint,
    ReduceLROnPlateau, TensorBoard
)

### Inisiasi Nilai Konstanta

In [ ]:
RANDOM_SEED   = 42
NUM_WORDS     = 10000
MAX_LENGTH    = 200
EMBED_DIM     = 128
EPOCH         = 20
BATCH_SIZE    = 32
OUTPUT_DIR = './models'
LOGS_DIR   = './logs'

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

## Load Dataset

In [ ]:
df = pd.read_excel(r"D:\Kuliah\DICODING\Capstone\HealMate_AI\DS\Empathetic Counseling Dataset\empathetic_counseling_final.xlsx")
print(f'Shape : {df.shape}')
df.head()

## Exploratory Data Analysis

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplikat : {df.duplicated().sum()}')
df.describe()

In [ ]:
# ── Distribusi Kelas ─────────────────────────────────────────
label_counts = df['predicted_emotion'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(label_counts.index, label_counts.values,
            color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title('Distribusi Kelas Emosi')
axes[0].set_xlabel('Emosi')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct='%1.1f%%', colors=['#e74c3c', '#3498db', '#2ecc71'],
            startangle=90)
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.show()
print(label_counts)

## Split Data

In [ ]:
X = df['input_clean'].fillna('').astype(str)
y = df['predicted_emotion']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Label encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls:12s} → {i}')

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_SEED, stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f'\nData split:')
print(f'  Train set : {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Val set   : {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test set  : {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)')

# Simpan index train untuk mapping retrieval
train_indices = X_train.index.tolist()
print(f'\n✅ Index train disimpan untuk mapping retrieval ({len(train_indices)} baris)')

## Tokenisasi dan Padding

In [ ]:
tokenizer = Tokenizer(num_words=NUM_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

vocab_size = min(len(tokenizer.word_index) + 1, NUM_WORDS)
print(f'Vocab size (aktual) : {len(tokenizer.word_index):,}')
print(f'Vocab size (capped) : {vocab_size:,}')

def texts_to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_train_seq = texts_to_padded(X_train)
X_val_seq   = texts_to_padded(X_val)
X_test_seq  = texts_to_padded(X_test)

print(f'\nShape setelah padding:')
print(f'  Train : {X_train_seq.shape}')
print(f'  Val   : {X_val_seq.shape}')
print(f'  Test  : {X_test_seq.shape}')

## Ekstraksi Fitur

### TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)
print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

### CountVectorizer

In [ ]:
count_vectorizer = CountVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_count = count_vectorizer.fit_transform(X_train)
X_val_count   = count_vectorizer.transform(X_val)
X_test_count  = count_vectorizer.transform(X_test)
print(f'Count matrix shape: {X_train_count.shape}')

## Modeling

### Helper Function

In [ ]:
def callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=5,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(f'{OUTPUT_DIR}/best_{model_name}.keras', monitor='val_accuracy',
                        save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                          min_lr=1e-6, verbose=1),
        TensorBoard(log_dir=f'{LOGS_DIR}/{model_name}', histogram_freq=0,
                    write_graph=False, update_freq='epoch'),
    ]


def train_model(model, model_name, X_train, y_train, X_val, y_val, epochs=EPOCH, batch_size=BATCH_SIZE):
    print(f'Training {model_name}...')
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks(model_name)
    )
    best_epoch = int(np.argmax(history.history['val_accuracy'])) + 1
    print(f'\n✅ {model_name} selesai di epoch {len(history.history["accuracy"])}, best epoch: {best_epoch}')
    return model, history


def evaluate_dl_model(model, X_test, y_test, model_name='Model'):
    y_pred = model.predict(X_test, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)

    acc = accuracy_score(y_test, y_pred_classes)
    f1  = f1_score(y_test, y_pred_classes, average='weighted')
    print(f'\n{model_name} — Accuracy: {acc:.4f}, F1-Score: {f1:.4f}')
    print(classification_report(y_test, y_pred_classes, target_names=le.classes_))

    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    return acc, f1


def evaluate_sklearn_model(model, X_test, y_test, model_name='Model'):
    y_pred_classes = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred_classes)
    f1  = f1_score(y_test, y_pred_classes, average='weighted')
    print(f'\n{model_name} — Accuracy: {acc:.4f}, F1-Score: {f1:.4f}')
    print(classification_report(y_test, y_pred_classes, target_names=le.classes_))

    cm = confusion_matrix(y_test, y_pred_classes)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{model_name} — Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

    return acc, f1


def plot_history(history, model_name='Model'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, metric, title in zip(axes,
        [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
        [f'{model_name} — Accuracy', f'{model_name} — Loss']):
        ax.plot(history.history[metric[0]], label='Train')
        ax.plot(history.history[metric[1]], label='Validation')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

print('✅ Helper functions ready!')

## Model Functions

In [ ]:
NUM_CLASSES = len(le.classes_)

def model_LogisticRegression():
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    model.fit(X_train_tfidf, y_train)
    return model

def model_LinearSVC():
    model = LinearSVC(random_state=RANDOM_SEED, max_iter=10000)
    model.fit(X_train_tfidf, y_train)
    return model

def model_MultinomialNB():
    model = MultinomialNB()
    model.fit(X_train_count, y_train)
    return model

def model_RandomForest():
    model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
    model.fit(X_train_tfidf, y_train)
    return model


def model_LSTM():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        LSTM(128, return_sequences=True),
        Dropout(0.5),
        LSTM(64),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def model_GRU():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        GRU(128, return_sequences=True),
        Dropout(0.5),
        GRU(64),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def model_CNN():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        Conv1D(128, kernel_size=5, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.5),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def model_BiLSTM():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        Bidirectional(LSTM(128, return_sequences=True)),
        Dropout(0.5),
        GlobalMaxPool1D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def model_CNN_BiLSTM():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        Conv1D(128, kernel_size=5, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.5),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

def model_CNN_BiLSTM_BatchNorm():
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LENGTH),
        Conv1D(128, kernel_size=5, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.5),
        Bidirectional(LSTM(64)),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

print('✅ Model functions ready!')

In [ ]:
# Logistic Regression
logreg_model = model_LogisticRegression()

print("="*50)
print(" Logistic Regression ")
print("="*50)

test_acc_logreg, test_f1_logreg = evaluate_sklearn_model(logreg_model, X_test_tfidf, y_test, "Logistic Regression")
print(f'Logistic Regression - Test Acc: {test_acc_logreg:.4f}, F1: {test_f1_logreg:.4f}')

In [ ]:
# LinearSVC
linear_svc_model = model_LinearSVC()

print("="*50)
print(" LinearSVC ")
print("="*50)

test_acc_linear_svc, test_f1_linear_svc = evaluate_sklearn_model(linear_svc_model, X_test_tfidf, y_test, "LinearSVC")
print(f'LinearSVC - Test Acc: {test_acc_linear_svc:.4f}, F1: {test_f1_linear_svc:.4f}')

In [ ]:
# Multinomial Naive Bayes
multinomial_nb_model = model_MultinomialNB()

print("="*50)
print(" Multinomial Naive Bayes ")
print("="*50)

test_acc_multinomial_nb, test_f1_multinomial_nb = evaluate_sklearn_model(multinomial_nb_model, X_test_count, y_test, "Multinomial Naive Bayes")
print(f'Multinomial Naive Bayes - Test Acc: {test_acc_multinomial_nb:.4f}, F1: {test_f1_multinomial_nb:.4f}')

In [ ]:
# Random Forest
RF_model = model_RandomForest()

print("="*50)
print(" Random Forest ")
print("="*50)

test_acc_RF, test_f1_RF = evaluate_sklearn_model(RF_model, X_test_tfidf, y_test, "Random Forest")
print(f'Random Forest - Test Acc: {test_acc_RF:.4f}, F1: {test_f1_RF:.4f}')

In [ ]:
# LSTM
lstm_model = model_LSTM()

print("="*50)
print(" LSTM ")
print("="*50)

lstm_model, lstm_history = train_model(lstm_model, 'LSTM', X_train_seq, y_train, X_val_seq, y_val)

plot_history(lstm_history, 'LSTM')

best_epoch_lstm  = int(np.argmax(lstm_history.history['val_accuracy'])) + 1
train_acc_lstm   = lstm_history.history['accuracy'][best_epoch_lstm-1]
val_acc_lstm     = lstm_history.history['val_accuracy'][best_epoch_lstm-1]
test_acc_lstm, _ = evaluate_dl_model(lstm_model, X_test_seq, y_test, "LSTM")

print(f'LSTM - Best Epoch: {best_epoch_lstm}, Train Acc: {train_acc_lstm:.4f}, Val Acc: {val_acc_lstm:.4f}, Test Acc: {test_acc_lstm:.4f}')

In [ ]:
# GRU
gru_model = model_GRU()

print("\n" + "="*50)
print(" GRU ")
print("="*50)

gru_model, gru_history = train_model(gru_model, 'GRU', X_train_seq, y_train, X_val_seq, y_val)

plot_history(gru_history, 'GRU')

best_epoch_gru  = int(np.argmax(gru_history.history['val_accuracy'])) + 1
train_acc_gru   = gru_history.history['accuracy'][best_epoch_gru-1]
val_acc_gru     = gru_history.history['val_accuracy'][best_epoch_gru-1]
test_acc_gru, _ = evaluate_dl_model(gru_model, X_test_seq, y_test, "GRU")

print(f'GRU - Best Epoch: {best_epoch_gru}, Train Acc: {train_acc_gru:.4f}, Val Acc: {val_acc_gru:.4f}, Test Acc: {test_acc_gru:.4f}')

In [ ]:
# CNN
cnn_model = model_CNN()

print("\n" + "="*50)
print(" CNN ")
print("="*50)

cnn_model, cnn_history = train_model(cnn_model, 'CNN', X_train_seq, y_train, X_val_seq, y_val)

plot_history(cnn_history, 'CNN')

best_epoch_cnn  = int(np.argmax(cnn_history.history['val_accuracy'])) + 1
train_acc_cnn   = cnn_history.history['accuracy'][best_epoch_cnn-1]
val_acc_cnn     = cnn_history.history['val_accuracy'][best_epoch_cnn-1]
test_acc_cnn, _ = evaluate_dl_model(cnn_model, X_test_seq, y_test, "CNN")

print(f'CNN - Best Epoch: {best_epoch_cnn}, Train Acc: {train_acc_cnn:.4f}, Val Acc: {val_acc_cnn:.4f}, Test Acc: {test_acc_cnn:.4f}')

In [ ]:
# BiLSTM
bilstm_model = model_BiLSTM()

print("\n" + "="*50)
print(" BiLSTM ")
print("="*50)

bilstm_model, bilstm_history = train_model(bilstm_model, 'BiLSTM', X_train_seq, y_train, X_val_seq, y_val)

plot_history(bilstm_history, 'BiLSTM')

best_epoch_bilstm  = int(np.argmax(bilstm_history.history['val_accuracy'])) + 1
train_acc_bilstm   = bilstm_history.history['accuracy'][best_epoch_bilstm-1]
val_acc_bilstm     = bilstm_history.history['val_accuracy'][best_epoch_bilstm-1]
test_acc_bilstm, _ = evaluate_dl_model(bilstm_model, X_test_seq, y_test, "BiLSTM")

print(f'BiLSTM - Best Epoch: {best_epoch_bilstm}, Train Acc: {train_acc_bilstm:.4f}, Val Acc: {val_acc_bilstm:.4f}, Test Acc: {test_acc_bilstm:.4f}')

In [ ]:
# CNN + BiLSTM
cnn_bilstm_model = model_CNN_BiLSTM()

print("\n" + "="*50)
print(" CNN + BiLSTM ")
print("="*50)

cnn_bilstm_model, cnn_bilstm_history = train_model(cnn_bilstm_model, 'CNN_BiLSTM', X_train_seq, y_train, X_val_seq, y_val)

plot_history(cnn_bilstm_history, 'CNN + BiLSTM')

best_epoch_cnn_bilstm  = int(np.argmax(cnn_bilstm_history.history['val_accuracy'])) + 1
train_acc_cnn_bilstm   = cnn_bilstm_history.history['accuracy'][best_epoch_cnn_bilstm-1]
val_acc_cnn_bilstm     = cnn_bilstm_history.history['val_accuracy'][best_epoch_cnn_bilstm-1]
test_acc_cnn_bilstm, _ = evaluate_dl_model(cnn_bilstm_model, X_test_seq, y_test, "CNN + BiLSTM")

print(f'CNN + BiLSTM - Best Epoch: {best_epoch_cnn_bilstm}, Train Acc: {train_acc_cnn_bilstm:.4f}, Val Acc: {val_acc_cnn_bilstm:.4f}, Test Acc: {test_acc_cnn_bilstm:.4f}')

In [ ]:
# CNN + BiLSTM + BatchNorm
cnn_bilstm_bn_model = model_CNN_BiLSTM_BatchNorm()

print("\n" + "="*50)
print(" CNN + BiLSTM + BatchNorm ")
print("="*50)

cnn_bilstm_bn_model, cnn_bilstm_bn_history = train_model(cnn_bilstm_bn_model, 'CNN_BiLSTM_BN', X_train_seq, y_train, X_val_seq, y_val)

plot_history(cnn_bilstm_bn_history, 'CNN + BiLSTM + BatchNorm')

best_epoch_cnn_bilstm_bn  = int(np.argmax(cnn_bilstm_bn_history.history['val_accuracy'])) + 1
train_acc_cnn_bilstm_bn   = cnn_bilstm_bn_history.history['accuracy'][best_epoch_cnn_bilstm_bn-1]
val_acc_cnn_bilstm_bn     = cnn_bilstm_bn_history.history['val_accuracy'][best_epoch_cnn_bilstm_bn-1]
test_acc_cnn_bilstm_bn, _ = evaluate_dl_model(cnn_bilstm_bn_model, X_test_seq, y_test, "CNN + BiLSTM + BatchNorm")

print(f'CNN+BiLSTM+BN - Best Epoch: {best_epoch_cnn_bilstm_bn}, Train Acc: {train_acc_cnn_bilstm_bn:.4f}, Val Acc: {val_acc_cnn_bilstm_bn:.4f}, Test Acc: {test_acc_cnn_bilstm_bn:.4f}')

## Evaluasi dan Metrics

In [ ]:
print("=" * 50)
print(" Evaluasi Akhir ")
print("=" * 50)

models_info = [
    ('Logistic Regression',        test_acc_logreg),
    ('LinearSVC',                  test_acc_linear_svc),
    ('Multinomial Naive Bayes',    test_acc_multinomial_nb),
    ('Random Forest',              test_acc_RF),
    ('LSTM',                       test_acc_lstm),
    ('GRU',                        test_acc_gru),
    ('CNN',                        test_acc_cnn),
    ('BiLSTM',                     test_acc_bilstm),
    ('CNN + BiLSTM',               test_acc_cnn_bilstm),
    ('CNN + BiLSTM + BatchNorm',   test_acc_cnn_bilstm_bn),
]

print(f'{"Model":35s} | {"Test Accuracy":>12s}')
print("-" * 50)

for name, acc in models_info:
    print(f'{name:35s} | {acc:.4f}')

best_model_info = max(models_info, key=lambda x: x[1])
print("\nModel terbaik:", best_model_info[0], f"({best_model_info[1]:.4f})")

## Perbandingan

In [ ]:
# Perbandingan performa model dan tampilkan model terbaik
df_results = pd.DataFrame(models_info, columns=['Model', 'TestAccuracy']).sort_values('TestAccuracy', ascending=False).reset_index(drop=True)

best = df_results.iloc[0]

plt.figure(figsize=(10, 6))
colors = ['#2ecc71' if m == best['Model'] else '#3498db' for m in df_results['Model']]
bars = plt.barh(df_results['Model'], df_results['TestAccuracy'], color=colors)
for i, acc in enumerate(df_results['TestAccuracy']):
    plt.text(acc + 0.002, i, f'{acc:.4f}', va='center')
plt.title('Perbandingan Test Accuracy Model', fontweight='bold')
plt.xlabel('Test Accuracy')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

print(f"Model terbaik: {best['Model']} ({best['TestAccuracy']:.4f})")